# Actor-Critic：TD 学习与在线策略更新

上一章介绍了 REINFORCE：用完整 episode 的 Monte Carlo 回报估计价值，再更新策略。

本章介绍 Actor-Critic。核心变化是使用 Temporal-Difference（TD，时序差分）误差进行 bootstrapping，从而可以在 episode 尚未结束时更新价值函数和策略。


## 1. 状态价值的递归关系

状态价值定义为

$$V^\pi(s)=\mathbb{E}[G_t\mid s_t=s].$$

由于

$$G_t=r_{t+1}+\gamma G_{t+1},$$

可得到 Bellman expectation equation：

$$V^\pi(s_t)=\mathbb{E}[r_{t+1}+\gamma V^\pi(s_{t+1})\mid s_t].$$

这个递归结构让我们无需等到整个 episode 结束就可以构造价值学习目标。


## 2. TD Target 与 TD Error

用参数化函数 $v_\phi(s)$ 近似真实状态价值。对一次实际转移 $(s_t,a_t,r_{t+1},s_{t+1})$，可构造 one-step TD target：

$$y_t=r_{t+1}+\gamma v_\phi(s_{t+1}).$$

TD error 定义为

$$\delta_t=y_t-v_\phi(s_t)=r_{t+1}+\gamma v_\phi(s_{t+1})-v_\phi(s_t).$$

使用当前估计 $v_\phi(s_{t+1})$ 去构造自己的学习目标，称为 **bootstrapping**。


## 3. 终止状态与时间截断

如果 $s_{t+1}$ 是真正的终止状态（`terminated=True`），其后续价值应为 0，因此

$$y_t=r_{t+1}.$$

如果只是达到时间上限而 `truncated=True`，通常不能简单把下一状态价值设为 0，因为任务本身未必真正结束。


## 4. TD Error 为什么可以用于更新 Actor

Advantage 定义为

$$A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s).$$

在 one-step TD 近似下，TD error 可以看作 Advantage 的一个样本估计：

$$\delta_t\approx A^\pi(s_t,a_t).$$

因此 Actor 可以使用

$$\delta_t\nabla_\theta\log\pi_\theta(a_t\mid s_t)$$

作为策略梯度方向。直观上，如果 $\delta_t>0$，说明当前动作的结果好于 Critic 原先预期，应提高这个动作的概率；反之则降低。


## 5. Actor 与 Critic 的职责

**Actor**：参数化策略 $\pi_\theta(a\mid s)$，负责选择动作。

**Critic**：参数化价值函数 $v_\phi(s)$，负责评价当前状态，并通过 TD error 为 Actor 提供学习信号。

典型损失为

$$L_{critic}=\frac{1}{2}\delta_t^2,$$

以及

$$L_{actor}=-\delta_t\log\pi_\theta(a_t\mid s_t).$$

实现 Actor loss 时通常需要对 $\delta_t$ `detach`，避免 Actor 更新反向改变 Critic。


In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, action_dim)
        )
    def distribution(self, state):
        return Categorical(logits=self.net(state))

class Critic(nn.Module):
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, 1)
        )
    def forward(self, state):
        return self.net(state).squeeze(-1)


## 6. 在线（Online）Actor-Critic

Online Actor-Critic 在每个环境 step 后立即更新：

1. 从 $\pi_\theta(\cdot\mid s_t)$ 采样动作；
2. 环境返回 $r_{t+1},s_{t+1}$；
3. 计算 TD target 与 TD error；
4. 更新 Critic；
5. 使用 TD error 更新 Actor；
6. 进入下一状态继续交互。

这种方式不需要等待 episode 结束，适用于连续任务，也能更及时地利用新数据。


In [ ]:
def actor_critic_step(actor, critic, actor_opt, critic_opt,
                      state, action, reward, next_state, terminated, gamma=0.99):
    value = critic(state)
    with torch.no_grad():
        next_value = torch.zeros_like(value) if terminated else critic(next_state)
        target = reward + gamma * next_value
    delta = target - value

    critic_loss = 0.5 * delta.pow(2).mean()
    critic_opt.zero_grad()
    critic_loss.backward()
    critic_opt.step()

    dist = actor.distribution(state)
    log_prob = dist.log_prob(action)
    actor_loss = -(delta.detach() * log_prob).mean()
    actor_opt.zero_grad()
    actor_loss.backward()
    actor_opt.step()
    return actor_loss.item(), critic_loss.item()


## 7. 批量式（Batch）Actor-Critic

也可以先收集一段轨迹，再把多个 transition 组成 batch 统一更新。这种方式更适合 GPU 并行，梯度方差通常也更稳定。

在线与批量不是不同的基本理论，而是数据收集和参数更新粒度不同。实际深度强化学习通常会在二者之间选择一个折中，例如每收集固定数量 step 后做若干次 mini-batch 更新。


## 8. Monte Carlo 与 TD 的偏差—方差权衡

REINFORCE 的 Monte Carlo target 基本无 bootstrapping 偏差，但因为使用未来整段随机回报，方差较大。

one-step TD target 使用当前价值估计，因此方差通常更小，但会引入由价值函数近似误差造成的偏差。

这构成强化学习中的经典 bias-variance trade-off。多步回报、$n$-step TD 和 GAE 等方法都可以看作在这两端之间做折中。


## 9. On-policy Actor-Critic

本章方法仍然是 on-policy：用于策略更新的数据来自当前策略。随着策略变化，旧数据不能无限重复使用。

下一章 TD3 则进入 off-policy Actor-Critic：通过动作价值函数、目标网络和 Replay Buffer，使过去的数据可以被重复利用。


## 10. 实践中应观察什么

训练 Actor-Critic 时，不要只看单次 episode 回报。建议同时记录：

- 滑动平均累计回报；
- Critic loss；
- TD error 的均值与方差；
- 策略熵（随机策略是否过早变得确定）；
- 不同随机种子的结果分布。

如果 Critic 明显失真，Actor 的更新方向也会随之失真，因此价值估计质量是 Actor-Critic 稳定训练的核心。


## 本章要点

- Bellman 递归关系允许使用 bootstrapping。
- TD error 同时是 Critic 的学习误差和 Advantage 的局部估计。
- Actor 负责行动，Critic 负责提供价值评价和学习信号。
- Online Actor-Critic 可以逐 step 学习，不需要等待 episode 结束。
- Monte Carlo 与 TD 存在偏差—方差权衡。
- 本章仍是 on-policy；下一章 TD3 将进入 off-policy 连续控制。

配套练习见 `ex2_actor_critic.ipynb`。
